> **Public release note.** Notebook outputs and execution counts have been removed because the underlying Malaysian Motor claims data are confidential. Local user-specific paths and record identifiers have also been removed. The code documents the analysis workflow but cannot be executed end-to-end without appropriately structured confidential input data.


# 05.9D — Two-Year Historical Pure IBNR and Integrated Framework

This notebook completes the 2020 Q4 to 2022 Q4 historical back-test by adding a rolling
pure IBNR allowance to the Random Forest and XGBoost reported-claim projections, with
a separate early-maturity macro treatment for AY2020. The completed estimates are then
compared with Incurred Chain Ladder on a like-for-like total-portfolio basis.

## Set up the two-year Integrated Framework analysis

This section loads the required Python packages, defines the historical input and output
folders, sets the three-comparator and eight-quarter assumptions, and checks that the
preceding two-year back-test outputs are available.

In [ ]:
from pathlib import Path
import gc
import json
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_FOLDER = Path(
    "/path/to/BI_large_claims_project"
)

HISTORICAL_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_9C_historical_valuation_diagonal_2yr"
)

OUTPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_9D_historical_pure_ibnr_integrated_framework_2yr"
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

ROLLING_COMPARATOR_YEARS = 3
EXPECTED_HORIZON_QTRS = 8
STREAM_BATCH_SIZE = 250_000

ONE_YEAR_INTEGRATED_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_9B_historical_pure_ibnr_integrated_framework"
)

required_historical_files = {
    "portfolio": HISTORICAL_FOLDER / "historical_diagonal_portfolio_summary.csv",
    "accident_year": HISTORICAL_FOLDER / "historical_diagonal_comparison_by_accident_year.csv",
    "scope": HISTORICAL_FOLDER / "historical_diagonal_outcome_scope_reconciliation.csv",
    "deployment": HISTORICAL_FOLDER / "historical_diagonal_deployment_audit.csv",
    "cl_factors": HISTORICAL_FOLDER / "historical_incurred_chain_ladder_factors.csv",
}

for name, path in required_historical_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name} input: {path}")

print("Historical input folder:", HISTORICAL_FOLDER)
print("Output folder:", OUTPUT_FOLDER)

## Set the master claims-development file

The confidential longitudinal claims file is assumed to be stored at
`processed/BI_large_claims_master_long.parquet`. The next cell defines this location
directly and checks that the file is present before the two-year analysis continues.

In [ ]:
MASTER_PATH = (
    PROJECT_FOLDER
    / "processed"
    / "BI_large_claims_master_long.parquet"
)

if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"Master file not found: {MASTER_PATH}"
    )

print("Master file:", MASTER_PATH)


## Load the completed two-year historical diagonal

This section loads the RF, XGBoost, Chain Ladder and scope-reconciliation outputs from
the preceding two-year back-test. It also recovers the common $T_0$ and $T_1$ dates so
the pure IBNR calculation uses exactly the same valuation period.

In [ ]:
portfolio_existing = pd.read_csv(required_historical_files["portfolio"])
ay_existing = pd.read_csv(required_historical_files["accident_year"])
scope_existing = pd.read_csv(required_historical_files["scope"])
deployment_existing = pd.read_csv(required_historical_files["deployment"])
cl_factors = pd.read_csv(required_historical_files["cl_factors"])

def quarter_label_to_index(label):
    year_text, quarter_text = str(label).strip().split()
    year = int(year_text)
    quarter = int(quarter_text.replace("Q", ""))
    return 4 * year + quarter

def quarter_index_to_label(index_value):
    index_value = int(index_value)
    year = (index_value - 1) // 4
    quarter = index_value - 4 * year
    return f"{year} Q{quarter}"

t0_labels = portfolio_existing["T0"].dropna().unique()
t1_labels = portfolio_existing["T1"].dropna().unique()

if len(t0_labels) != 1 or len(t1_labels) != 1:
    raise ValueError("Historical portfolio summary does not contain unique T0/T1 dates.")

T0_LABEL = str(t0_labels[0])
T1_LABEL = str(t1_labels[0])
T0_INDEX = quarter_label_to_index(T0_LABEL)
T1_INDEX = quarter_label_to_index(T1_LABEL)
HORIZON_QTRS = T1_INDEX - T0_INDEX

if HORIZON_QTRS != EXPECTED_HORIZON_QTRS:
    raise ValueError(
        f"This notebook requires an {EXPECTED_HORIZON_QTRS}-quarter horizon, "
        f"but the historical diagonal contains {HORIZON_QTRS} quarters."
    )

if T0_LABEL != "2020 Q4" or T1_LABEL != "2022 Q4":
    raise ValueError(
        "Unexpected historical dates. "
        f"Expected 2020 Q4 to 2022 Q4, obtained {T0_LABEL} to {T1_LABEL}."
    )

MIN_COMPARATOR_LAG_YEARS = int(np.ceil(HORIZON_QTRS / 4))
COMPARATOR_LAGS = list(
    range(
        MIN_COMPARATOR_LAG_YEARS,
        MIN_COMPARATOR_LAG_YEARS + ROLLING_COMPARATOR_YEARS,
    )
)

TARGET_AYS = sorted(ay_existing["ACC_YEAR"].astype(int).unique())
CURRENT_AY = (T0_INDEX - 1) // 4

print("T0:", T0_LABEL, T0_INDEX)
print("T1:", T1_LABEL, T1_INDEX)
print("Main diagonal AY range:", min(TARGET_AYS), "to", max(TARGET_AYS))
print("Current accident year requiring early treatment:", CURRENT_AY)

print("Comparator accident-year lags:", COMPARATOR_LAGS)


## Define the historical comparator dates

The eight-quarter horizon requires comparator accident years that are at least two years
older than the target year. This section constructs the three same-maturity comparator
snapshot/outcome pairs and checks that every comparator outcome was already observable
by $T_0$, preventing future-data leakage.

In [ ]:
historical_snapshot_indices = [
    T0_INDEX - 4 * lag
    for lag in COMPARATOR_LAGS
]
historical_outcome_indices = [
    snapshot_index + HORIZON_QTRS
    for snapshot_index in historical_snapshot_indices
]

REQUIRED_POSITION_INDICES = sorted(
    set(
        [T0_INDEX, T1_INDEX]
        + historical_snapshot_indices
        + historical_outcome_indices
    )
)

required_dates_df = pd.DataFrame({
    "valuation_index": REQUIRED_POSITION_INDICES,
    "valuation_label": [
        quarter_index_to_label(index_value)
        for index_value in REQUIRED_POSITION_INDICES
    ],
})

display(required_dates_df)

if max(historical_outcome_indices) > T0_INDEX:
    raise ValueError(
        "Comparator outcomes extend beyond T0 and would introduce leakage."
    )

if len(historical_snapshot_indices) != ROLLING_COMPARATOR_YEARS:
    raise ValueError("Unexpected number of comparator snapshot dates.")

## Read the master claims history in batches

This section reads only the fields needed to identify first-report dates and BI Excess
positions, processing the large master file in batches to control memory use. A claim is
treated as reported from the first valuation quarter with `CLAIMS_CNT > 0`, allowing
reported development and pure IBNR to be separated at each historical date.

In [ ]:
parquet_file = pq.ParquetFile(MASTER_PATH)
available_columns = set(parquet_file.schema_arrow.names)

REQUIRED_MASTER_COLUMNS = [
    "SOURCE_FILE",
    "CLAIMS_KEY",
    "ACC_YEAR",
    "ACC_QTR",
    "DEV_QTR",
    "CLAIMS_CNT",
    "CUM_INC_LARGE",
]

missing_columns = sorted(
    set(REQUIRED_MASTER_COLUMNS).difference(available_columns)
)
if missing_columns:
    raise ValueError(
        f"Master parquet is missing required fields: {missing_columns}"
    )

first_report_batches = []
position_batches = {
    valuation_index: []
    for valuation_index in REQUIRED_POSITION_INDICES
}

rows_scanned = 0

for batch_number, record_batch in enumerate(
    parquet_file.iter_batches(
        batch_size=STREAM_BATCH_SIZE,
        columns=REQUIRED_MASTER_COLUMNS,
    ),
    start=1,
):
    chunk = record_batch.to_pandas()
    rows_scanned += len(chunk)

    for col in ["ACC_YEAR", "ACC_QTR", "DEV_QTR", "CLAIMS_CNT", "CUM_INC_LARGE"]:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk["CLAIMS_CNT"] = chunk["CLAIMS_CNT"].fillna(0.0)
    chunk["CUM_INC_LARGE"] = chunk["CUM_INC_LARGE"].fillna(0.0)

    chunk["ACCIDENT_QTR_INDEX"] = (
        4 * chunk["ACC_YEAR"].astype(int)
        + chunk["ACC_QTR"].astype(int)
    )
    chunk["VALUATION_QTR_INDEX"] = (
        chunk["ACCIDENT_QTR_INDEX"]
        + chunk["DEV_QTR"].astype(int)
    )

    report_rows = chunk.loc[
        chunk["CLAIMS_CNT"] > 0,
        [
            "SOURCE_FILE",
            "CLAIMS_KEY",
            "ACC_YEAR",
            "ACC_QTR",
            "VALUATION_QTR_INDEX",
        ],
    ]

    if not report_rows.empty:
        first_report_batch = (
            report_rows
            .groupby(
                ["SOURCE_FILE", "CLAIMS_KEY", "ACC_YEAR", "ACC_QTR"],
                as_index=False,
            )
            .agg(first_report_index=("VALUATION_QTR_INDEX", "min"))
        )
        first_report_batches.append(first_report_batch)

    required_rows = chunk.loc[
        chunk["VALUATION_QTR_INDEX"].isin(REQUIRED_POSITION_INDICES),
        [
            "SOURCE_FILE",
            "CLAIMS_KEY",
            "ACC_YEAR",
            "ACC_QTR",
            "VALUATION_QTR_INDEX",
            "CUM_INC_LARGE",
            "CLAIMS_CNT",
        ],
    ]

    if not required_rows.empty:
        for valuation_index, date_rows in required_rows.groupby(
            "VALUATION_QTR_INDEX"
        ):
            position_batches[int(valuation_index)].append(
                date_rows.copy()
            )

    del chunk, report_rows, required_rows, record_batch

    if batch_number % 10 == 0:
        gc.collect()
        print(f"Scanned {rows_scanned:,} master rows...")

if not first_report_batches:
    raise ValueError("No reported claims were identified in the master data.")

first_report = pd.concat(first_report_batches, ignore_index=True)
first_report = (
    first_report
    .groupby(
        ["SOURCE_FILE", "CLAIMS_KEY", "ACC_YEAR", "ACC_QTR"],
        as_index=False,
    )
    .agg(first_report_index=("first_report_index", "min"))
)

del first_report_batches
gc.collect()

positions = {}

for valuation_index in REQUIRED_POSITION_INDICES:
    if not position_batches[valuation_index]:
        raise ValueError(
            f"No rows were found for valuation index {valuation_index} "
            f"({quarter_index_to_label(valuation_index)})."
        )

    position_df = pd.concat(
        position_batches[valuation_index],
        ignore_index=True,
    )

    position_df = (
        position_df
        .sort_values(
            [
                "SOURCE_FILE",
                "CLAIMS_KEY",
                "VALUATION_QTR_INDEX",
            ]
        )
        .drop_duplicates(
            ["SOURCE_FILE", "CLAIMS_KEY"],
            keep="last",
        )
    )

    position_df = position_df.merge(
        first_report[
            [
                "SOURCE_FILE",
                "CLAIMS_KEY",
                "first_report_index",
            ]
        ],
        on=["SOURCE_FILE", "CLAIMS_KEY"],
        how="left",
        validate="one_to_one",
    )

    position_df["reported_by_position"] = (
        position_df["first_report_index"].notna()
        & (position_df["first_report_index"] <= valuation_index)
    )

    positions[valuation_index] = position_df

    print(
        quarter_index_to_label(valuation_index),
        "position rows:",
        f"{len(position_df):,}",
        "| reported:",
        f"{int(position_df['reported_by_position'].sum()):,}",
        "| BIXS:",
        f"{position_df['CUM_INC_LARGE'].sum():,.2f}",
    )

del position_batches
gc.collect()

## Define the cohort measurement function

This function measures an accident-year cohort between a chosen snapshot and outcome
date. It separates development on claims already reported at the snapshot, later-reported
pure IBNR BI Excess, and total BI Excess development over the eight-quarter horizon.

In [ ]:
KEY_COLUMNS = ["SOURCE_FILE", "CLAIMS_KEY"]

def cohort_metrics(accident_year, snapshot_index, outcome_index):
    snapshot = positions[snapshot_index].loc[
        positions[snapshot_index]["ACC_YEAR"].eq(accident_year)
    ].copy()

    outcome = positions[outcome_index].loc[
        positions[outcome_index]["ACC_YEAR"].eq(accident_year)
    ].copy()

    cohort_reports = first_report.loc[
        first_report["ACC_YEAR"].eq(accident_year)
    ].copy()

    reported_keys = cohort_reports.loc[
        cohort_reports["first_report_index"] <= snapshot_index,
        KEY_COLUMNS,
    ].drop_duplicates()

    reported_claim_count = len(reported_keys)

    snapshot_reported = snapshot.merge(
        reported_keys.assign(reported_at_snapshot=True),
        on=KEY_COLUMNS,
        how="inner",
    )

    outcome_with_report = outcome.merge(
        cohort_reports[
            KEY_COLUMNS + ["first_report_index"]
        ],
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
        suffixes=("", "_report"),
    )

    reported_outcome = outcome_with_report.loc[
        outcome_with_report["first_report_index"] <= snapshot_index
    ]

    pure_ibnr_outcome = outcome_with_report.loc[
        (outcome_with_report["first_report_index"] > snapshot_index)
        & (outcome_with_report["first_report_index"] <= outcome_index)
        & (outcome_with_report["CUM_INC_LARGE"] > 0)
    ]

    late_reported_all = outcome_with_report.loc[
        (outcome_with_report["first_report_index"] > snapshot_index)
        & (outcome_with_report["first_report_index"] <= outcome_index)
    ]

    snapshot_reported_bixs = float(
        snapshot_reported["CUM_INC_LARGE"].sum()
    )
    reported_outcome_bixs = float(
        reported_outcome["CUM_INC_LARGE"].sum()
    )
    full_outcome_bixs = float(
        outcome_with_report["CUM_INC_LARGE"].sum()
    )
    pure_ibnr_bixs = float(
        pure_ibnr_outcome["CUM_INC_LARGE"].sum()
    )

    return {
        "accident_year": int(accident_year),
        "snapshot_index": int(snapshot_index),
        "snapshot_label": quarter_index_to_label(snapshot_index),
        "outcome_index": int(outcome_index),
        "outcome_label": quarter_index_to_label(outcome_index),
        "reported_claim_count_at_snapshot": int(reported_claim_count),
        "snapshot_reported_bixs": snapshot_reported_bixs,
        "reported_claim_bixs_at_outcome": reported_outcome_bixs,
        "reported_claim_future_development": (
            reported_outcome_bixs - snapshot_reported_bixs
        ),
        "pure_ibnr_positive_bixs_claim_count": int(
            len(pure_ibnr_outcome)
        ),
        "pure_ibnr_bixs_amount": pure_ibnr_bixs,
        "late_reported_claim_count_all_outcomes": int(
            len(late_reported_all)
        ),
        "full_portfolio_bixs_at_outcome": full_outcome_bixs,
        "total_future_development": (
            full_outcome_bixs - snapshot_reported_bixs
        ),
    }

## Build same-maturity rolling comparator experience

For each target accident year, this section selects three older comparator accident years
using lags of two, three and four years. Each comparator is observed at the same relative
maturity and followed for eight quarters, with all outcomes known by $T_0$.

In [ ]:
all_target_ays = sorted(set(TARGET_AYS + [CURRENT_AY]))
minimum_data_ay = int(first_report["ACC_YEAR"].min())

comparator_rows = []

for target_ay in all_target_ays:
    for lag in COMPARATOR_LAGS:
        comparator_ay = target_ay - lag

        if comparator_ay < minimum_data_ay:
            continue

        snapshot_index = T0_INDEX - 4 * lag
        outcome_index = snapshot_index + HORIZON_QTRS

        if outcome_index > T0_INDEX:
            raise ValueError(
                "Comparator outcome occurs after T0 and introduces leakage."
            )

        metrics = cohort_metrics(
            comparator_ay,
            snapshot_index,
            outcome_index,
        )

        comparator_rows.append({
            "target_accident_year": target_ay,
            "comparator_lag_years": lag,
            "comparator_accident_year": comparator_ay,
            **metrics,
        })

comparator_detail = pd.DataFrame(comparator_rows)

comparator_detail.to_csv(
    OUTPUT_FOLDER / "historical_pure_ibnr_comparator_detail.csv",
    index=False,
)

display(comparator_detail.tail(12))

## Estimate pure IBNR by accident year

The three comparator cohorts are pooled to estimate the late-BI-Excess frequency and
average severity:

$$
\widehat{f}_{\mathrm{AY}}
=
\frac{
\sum \text{late positive-BIXS claims}
}{
\sum \text{reported claims at comparator snapshots}
}
$$

$$
\widehat{s}_{\mathrm{AY}}
=
\frac{
\sum \text{late-reported BI Excess amount}
}{
\sum \text{late positive-BIXS claims}
}
$$

The pure IBNR allowance for the target accident year is then:

$$
\widehat{U}^{\mathrm{Pure\ IBNR}}_{\mathrm{AY}}
=
N^{\mathrm{Reported}}_{\mathrm{AY},T_0}
\widehat{f}_{\mathrm{AY}}
\widehat{s}_{\mathrm{AY}}
$$

In [ ]:
pure_ibnr_estimate_rows = []

for target_ay in all_target_ays:
    target_metrics = cohort_metrics(
        target_ay,
        T0_INDEX,
        T1_INDEX,
    )

    comparators = comparator_detail.loc[
        comparator_detail["target_accident_year"].eq(target_ay)
    ].copy()

    comparator_exposure = float(
        comparators["reported_claim_count_at_snapshot"].sum()
    )
    comparator_pure_count = float(
        comparators["pure_ibnr_positive_bixs_claim_count"].sum()
    )
    comparator_pure_amount = float(
        comparators["pure_ibnr_bixs_amount"].sum()
    )

    if comparator_exposure > 0:
        frequency = comparator_pure_count / comparator_exposure
        amount_per_reported_claim = (
            comparator_pure_amount / comparator_exposure
        )
    else:
        frequency = 0.0
        amount_per_reported_claim = 0.0

    severity = (
        comparator_pure_amount / comparator_pure_count
        if comparator_pure_count > 0
        else 0.0
    )

    estimated_count = (
        target_metrics["reported_claim_count_at_snapshot"]
        * frequency
    )
    estimated_amount = (
        target_metrics["reported_claim_count_at_snapshot"]
        * amount_per_reported_claim
    )

    actual_amount = target_metrics["pure_ibnr_bixs_amount"]

    pure_ibnr_estimate_rows.append({
        "ACC_YEAR": target_ay,
        "T0": T0_LABEL,
        "T1": T1_LABEL,
        "comparator_accident_years": ",".join(
            str(int(value))
            for value in comparators[
                "comparator_accident_year"
            ].tolist()
        ),
        "comparator_year_count": len(comparators),
        "target_reported_claim_count_at_T0": (
            target_metrics["reported_claim_count_at_snapshot"]
        ),
        "comparator_reported_claim_exposure": comparator_exposure,
        "comparator_pure_ibnr_positive_claim_count": (
            comparator_pure_count
        ),
        "comparator_pure_ibnr_amount": comparator_pure_amount,
        "estimated_pure_ibnr_frequency": frequency,
        "estimated_pure_ibnr_severity": severity,
        "estimated_pure_ibnr_amount_per_reported_claim": (
            amount_per_reported_claim
        ),
        "estimated_pure_ibnr_claim_count": estimated_count,
        "estimated_pure_ibnr_bixs_amount": estimated_amount,
        "observed_pure_ibnr_positive_claim_count_T1": (
            target_metrics["pure_ibnr_positive_bixs_claim_count"]
        ),
        "observed_pure_ibnr_bixs_amount_T1": actual_amount,
        "pure_ibnr_amount_error": estimated_amount - actual_amount,
        "pure_ibnr_absolute_error": abs(estimated_amount - actual_amount),
    })

pure_ibnr_by_ay = pd.DataFrame(pure_ibnr_estimate_rows)

pure_ibnr_by_ay.to_csv(
    OUTPUT_FOLDER / "historical_pure_ibnr_estimate_by_accident_year.csv",
    index=False,
)

display(pure_ibnr_by_ay)

## Reconcile the observed pure IBNR amount

This section checks that the newly reconstructed pure IBNR monetary amounts agree with
the earlier scope-reconciliation results. Any difference is flagged before the
allowance is added to the machine-learning projections.

In [ ]:
pure_main = pure_ibnr_by_ay.loc[
    pure_ibnr_by_ay["ACC_YEAR"].isin(TARGET_AYS)
].copy()

pure_reconciliation = pure_main.merge(
    scope_existing[
        [
            "ACC_YEAR",
            "observed_pure_ibnr_bixs_T1",
            "observed_pure_ibnr_claim_count",
        ]
    ],
    on="ACC_YEAR",
    how="left",
    validate="one_to_one",
)

pure_reconciliation["amount_difference_new_minus_existing"] = (
    pure_reconciliation["observed_pure_ibnr_bixs_amount_T1"]
    - pure_reconciliation["observed_pure_ibnr_bixs_T1"]
)

max_amount_difference = float(
    pure_reconciliation[
        "amount_difference_new_minus_existing"
    ].abs().max()
)

print(
    "Maximum pure IBNR amount reconciliation difference:",
    f"{max_amount_difference:,.2f}",
)

if max_amount_difference > 1.0:
    warnings.warn(
        "The newly derived pure IBNR amount differs from the earlier "
        "scope reconciliation. Review the reporting-date definition."
    )

pure_reconciliation.to_csv(
    OUTPUT_FOLDER / "historical_pure_ibnr_reconciliation_audit.csv",
    index=False,
)

display(
    pure_reconciliation[
        [
            "ACC_YEAR",
            "estimated_pure_ibnr_bixs_amount",
            "observed_pure_ibnr_bixs_amount_T1",
            "observed_pure_ibnr_positive_claim_count_T1",
            "observed_pure_ibnr_claim_count",
            "amount_difference_new_minus_existing",
        ]
    ]
)

## Add pure IBNR to Random Forest and XGBoost

For the main diagonal accident years, the reported-claim projection is combined with the
separate pure IBNR allowance:

$$
\widehat{U}^{M,\mathrm{Integrated}}_{\mathrm{AY}}
=
\widehat{U}^{M,\mathrm{Reported}}_{\mathrm{AY}}
+
\widehat{U}^{\mathrm{Pure\ IBNR}}_{\mathrm{AY}}
$$

where $M$ is Random Forest or XGBoost. This produces a total-portfolio estimate that can
be compared with Incurred Chain Ladder on the same basis.

In [ ]:
integrated_ay = ay_existing.merge(
    pure_main[
        [
            "ACC_YEAR",
            "estimated_pure_ibnr_bixs_amount",
            "observed_pure_ibnr_bixs_amount_T1",
        ]
    ],
    on="ACC_YEAR",
    how="left",
    validate="one_to_one",
)

integrated_ay["estimated_pure_ibnr_bixs_amount"] = (
    integrated_ay["estimated_pure_ibnr_bixs_amount"].fillna(0.0)
)

integrated_ay["rf_integrated_projected_latest_bixs"] = (
    integrated_ay["rf_projected_latest_bixs"]
    + integrated_ay["estimated_pure_ibnr_bixs_amount"]
)
integrated_ay["xgb_integrated_projected_latest_bixs"] = (
    integrated_ay["xgb_projected_latest_bixs"]
    + integrated_ay["estimated_pure_ibnr_bixs_amount"]
)

for method in ["rf", "xgb"]:
    projected_col = f"{method}_integrated_projected_latest_bixs"
    error_col = f"{method}_integrated_total_error"

    integrated_ay[error_col] = (
        integrated_ay[projected_col]
        - integrated_ay["full_portfolio_actual_latest_bixs_T1"]
    )
    integrated_ay[f"{method}_integrated_total_absolute_error"] = (
        integrated_ay[error_col].abs()
    )
    integrated_ay[f"{method}_integrated_total_percentage_error"] = np.where(
        integrated_ay["full_portfolio_actual_latest_bixs_T1"] != 0,
        integrated_ay[error_col]
        / integrated_ay["full_portfolio_actual_latest_bixs_T1"],
        np.nan,
    )

integrated_ay.to_csv(
    OUTPUT_FOLDER / "historical_integrated_framework_by_accident_year.csv",
    index=False,
)

display(
    integrated_ay[
        [
            "ACC_YEAR",
            "rf_projected_latest_bixs",
            "estimated_pure_ibnr_bixs_amount",
            "rf_integrated_projected_latest_bixs",
            "xgb_integrated_projected_latest_bixs",
            "cl_projected_latest_bixs_T1",
            "full_portfolio_actual_latest_bixs_T1",
            "rf_integrated_total_error",
            "xgb_integrated_total_error",
            "cl_total_error",
        ]
    ]
)

## Summarise the main Integrated Framework results

This section aggregates the accident-year Integrated Framework results to portfolio level for Random Forest, XGBoost and Incurred Chain Ladder. It reports the total projection, error, bias ratio and accident-year WAPE on the shared total-portfolio basis.


In [ ]:
def summarise_method(
    method,
    scope,
    projected_column,
    observed_column,
    absolute_error_column,
):
    observed = float(integrated_ay[observed_column].sum())
    projected = float(integrated_ay[projected_column].sum())
    error = projected - observed
    ay_wape = (
        float(integrated_ay[absolute_error_column].sum()) / abs(observed)
        if observed != 0
        else np.nan
    )

    return {
        "method": method,
        "scope": scope,
        "T0": T0_LABEL,
        "T1": T1_LABEL,
        "accident_year_min": int(integrated_ay["ACC_YEAR"].min()),
        "accident_year_max": int(integrated_ay["ACC_YEAR"].max()),
        "observed_latest_bixs_T1": observed,
        "projected_latest_bixs_T1": projected,
        "difference_projected_minus_observed": error,
        "bias_ratio": projected / observed if observed != 0 else np.nan,
        "accident_year_wape": ay_wape,
    }

integrated_portfolio_summary = pd.DataFrame([
    summarise_method(
        "Random Forest Integrated Framework",
        "Total portfolio: reported claims plus estimated pure IBNR",
        "rf_integrated_projected_latest_bixs",
        "full_portfolio_actual_latest_bixs_T1",
        "rf_integrated_total_absolute_error",
    ),
    summarise_method(
        "XGBoost Integrated Framework",
        "Total portfolio: reported claims plus estimated pure IBNR",
        "xgb_integrated_projected_latest_bixs",
        "full_portfolio_actual_latest_bixs_T1",
        "xgb_integrated_total_absolute_error",
    ),
    summarise_method(
        "Incurred Chain Ladder",
        "Total portfolio",
        "cl_projected_latest_bixs_T1",
        "full_portfolio_actual_latest_bixs_T1",
        "cl_total_absolute_error",
    ),
])

integrated_portfolio_summary.to_csv(
    OUTPUT_FOLDER / "historical_integrated_framework_portfolio_summary.csv",
    index=False,
)

display(integrated_portfolio_summary)

## Estimate the early-maturity current accident year

AY2020 is below DEV_QTR_4 at $T_0$, so neither claim-level model is applied to it.
Instead, the same three-comparator approach estimates total future development per
reported claim, including both reported development and pure IBNR:

$$
\text{Total future development}
=
\text{full BI Excess at outcome}
-
\text{reported BI Excess at snapshot}
$$

In [ ]:
early_comparators = comparator_detail.loc[
    comparator_detail["target_accident_year"].eq(CURRENT_AY)
].copy()

early_target_metrics = cohort_metrics(
    CURRENT_AY,
    T0_INDEX,
    T1_INDEX,
)

early_comparator_exposure = float(
    early_comparators["reported_claim_count_at_snapshot"].sum()
)
early_comparator_total_future = float(
    early_comparators["total_future_development"].sum()
)
early_comparator_reported_future = float(
    early_comparators["reported_claim_future_development"].sum()
)
early_comparator_pure_ibnr = float(
    early_comparators["pure_ibnr_bixs_amount"].sum()
)

early_amount_per_reported_claim = (
    early_comparator_total_future / early_comparator_exposure
    if early_comparator_exposure > 0
    else 0.0
)

early_estimated_future = (
    early_target_metrics["reported_claim_count_at_snapshot"]
    * early_amount_per_reported_claim
)
early_projected_latest = (
    early_target_metrics["snapshot_reported_bixs"]
    + early_estimated_future
)
early_observed_latest = (
    early_target_metrics["full_portfolio_bixs_at_outcome"]
)

early_macro_result = pd.DataFrame([{
    "ACC_YEAR": CURRENT_AY,
    "T0": T0_LABEL,
    "T1": T1_LABEL,
    "comparator_accident_years": ",".join(
        str(int(value))
        for value in early_comparators[
            "comparator_accident_year"
        ].tolist()
    ),
    "target_reported_claim_count_at_T0": (
        early_target_metrics["reported_claim_count_at_snapshot"]
    ),
    "target_reported_bixs_at_T0": (
        early_target_metrics["snapshot_reported_bixs"]
    ),
    "comparator_reported_claim_exposure": early_comparator_exposure,
    "comparator_total_future_development": (
        early_comparator_total_future
    ),
    "comparator_reported_claim_future_development": (
        early_comparator_reported_future
    ),
    "comparator_pure_ibnr_bixs": early_comparator_pure_ibnr,
    "estimated_total_future_per_reported_claim": (
        early_amount_per_reported_claim
    ),
    "estimated_total_future_development": early_estimated_future,
    "macro_projected_latest_bixs": early_projected_latest,
    "observed_latest_bixs_T1": early_observed_latest,
    "difference_projected_minus_observed": (
        early_projected_latest - early_observed_latest
    ),
    "bias_ratio": (
        early_projected_latest / early_observed_latest
        if early_observed_latest != 0
        else np.nan
    ),
    "observed_reported_claim_future_development": (
        early_target_metrics["reported_claim_future_development"]
    ),
    "observed_pure_ibnr_bixs": (
        early_target_metrics["pure_ibnr_bixs_amount"]
    ),
    "observed_total_future_development": (
        early_target_metrics["total_future_development"]
    ),
}])

early_macro_result.to_csv(
    OUTPUT_FOLDER / "historical_early_maturity_macro_current_ay.csv",
    index=False,
)

display(early_macro_result.T)

## Extend Incurred Chain Ladder to the current accident year

This section applies the existing historical Chain Ladder factors to each AY2020 accident
quarter from its maturity at $T_0$ to its maturity at $T_1$. This provides the corresponding
Chain Ladder benchmark for the early-maturity accident year.

In [ ]:
factor_lookup = (
    cl_factors
    .set_index("development_quarter")["factor"]
    .to_dict()
)
usable_lookup = (
    cl_factors
    .set_index("development_quarter")["factor_usable"]
    .to_dict()
)

t0_current_position = positions[T0_INDEX].loc[
    positions[T0_INDEX]["ACC_YEAR"].eq(CURRENT_AY)
].copy()

current_by_accident_quarter = (
    t0_current_position
    .groupby("ACC_QTR", as_index=False)
    .agg(incurred_bixs_T0=("CUM_INC_LARGE", "sum"))
)

current_cl_rows = []

for row in current_by_accident_quarter.itertuples(index=False):
    origin_index = 4 * CURRENT_AY + int(row.ACC_QTR)
    current_dev = T0_INDEX - origin_index
    target_dev = T1_INDEX - origin_index

    projected = float(row.incurred_bixs_T0)
    unusable_factor_count = 0

    for dev in range(current_dev, target_dev):
        factor = float(factor_lookup.get(dev, 1.0))
        if not bool(usable_lookup.get(dev, False)):
            unusable_factor_count += 1
        projected *= factor

    current_cl_rows.append({
        "ACC_YEAR": CURRENT_AY,
        "ACC_QTR": int(row.ACC_QTR),
        "development_at_T0": int(current_dev),
        "target_development_at_T1": int(target_dev),
        "incurred_bixs_T0": float(row.incurred_bixs_T0),
        "cl_projected_latest_bixs_T1": projected,
        "unusable_factor_count": unusable_factor_count,
    })

current_cl_projection = pd.DataFrame(current_cl_rows)

current_cl_projection.to_csv(
    OUTPUT_FOLDER / "historical_current_ay_cl_projection_by_accident_quarter.csv",
    index=False,
)

CURRENT_AY_CL_PROJECTED = float(
    current_cl_projection["cl_projected_latest_bixs_T1"].sum()
)

display(current_cl_projection)
print(
    "Current AY Chain Ladder projection:",
    f"{CURRENT_AY_CL_PROJECTED:,.2f}",
)

## Build the complete accident-year comparison

This section combines the Integrated Framework results for the model-supported accident
years with the early-maturity macro estimate for AY2020. It then calculates RF, XGBoost
and Chain Ladder errors against the full observed BI Excess outcome for every accident year.

In [ ]:
complete_ay = integrated_ay[
    [
        "ACC_YEAR",
        "full_portfolio_actual_latest_bixs_T1",
        "rf_integrated_projected_latest_bixs",
        "xgb_integrated_projected_latest_bixs",
        "cl_projected_latest_bixs_T1",
    ]
].copy()

current_complete_row = pd.DataFrame([{
    "ACC_YEAR": CURRENT_AY,
    "full_portfolio_actual_latest_bixs_T1": early_observed_latest,
    "rf_integrated_projected_latest_bixs": early_projected_latest,
    "xgb_integrated_projected_latest_bixs": early_projected_latest,
    "cl_projected_latest_bixs_T1": CURRENT_AY_CL_PROJECTED,
}])

complete_ay = pd.concat(
    [complete_ay, current_complete_row],
    ignore_index=True,
).sort_values("ACC_YEAR")

complete_ay["integrated_component"] = np.where(
    complete_ay["ACC_YEAR"].eq(CURRENT_AY),
    "Early-maturity macro",
    "Claim-level model plus pure IBNR",
)

for method in ["rf", "xgb"]:
    projected_col = f"{method}_integrated_projected_latest_bixs"
    complete_ay[f"{method}_complete_error"] = (
        complete_ay[projected_col]
        - complete_ay["full_portfolio_actual_latest_bixs_T1"]
    )
    complete_ay[f"{method}_complete_absolute_error"] = (
        complete_ay[f"{method}_complete_error"].abs()
    )
    complete_ay[f"{method}_complete_percentage_error"] = np.where(
        complete_ay["full_portfolio_actual_latest_bixs_T1"] != 0,
        complete_ay[f"{method}_complete_error"]
        / complete_ay["full_portfolio_actual_latest_bixs_T1"],
        np.nan,
    )

complete_ay["cl_complete_error"] = (
    complete_ay["cl_projected_latest_bixs_T1"]
    - complete_ay["full_portfolio_actual_latest_bixs_T1"]
)
complete_ay["cl_complete_absolute_error"] = (
    complete_ay["cl_complete_error"].abs()
)
complete_ay["cl_complete_percentage_error"] = np.where(
    complete_ay["full_portfolio_actual_latest_bixs_T1"] != 0,
    complete_ay["cl_complete_error"]
    / complete_ay["full_portfolio_actual_latest_bixs_T1"],
    np.nan,
)

complete_ay.to_csv(
    OUTPUT_FOLDER / "historical_complete_framework_by_accident_year.csv",
    index=False,
)

display(complete_ay)

## Summarise the complete portfolio results

The accident-year results are aggregated to portfolio level for the Random Forest
Integrated Framework, XGBoost Integrated Framework and Incurred Chain Ladder. The
summary reports the projected total, error, bias ratio and accident-year WAPE for each method.

In [ ]:
def complete_summary(method, projected_col, absolute_error_col):
    observed = float(
        complete_ay["full_portfolio_actual_latest_bixs_T1"].sum()
    )
    projected = float(complete_ay[projected_col].sum())

    return {
        "method": method,
        "scope": (
            f"Complete AY{int(complete_ay['ACC_YEAR'].min())}–"
            f"AY{int(complete_ay['ACC_YEAR'].max())} portfolio"
        ),
        "T0": T0_LABEL,
        "T1": T1_LABEL,
        "observed_latest_bixs_T1": observed,
        "projected_latest_bixs_T1": projected,
        "difference_projected_minus_observed": projected - observed,
        "bias_ratio": projected / observed if observed != 0 else np.nan,
        "accident_year_wape": (
            float(complete_ay[absolute_error_col].sum()) / abs(observed)
            if observed != 0
            else np.nan
        ),
    }

complete_portfolio_summary = pd.DataFrame([
    complete_summary(
        "Random Forest Integrated Framework",
        "rf_integrated_projected_latest_bixs",
        "rf_complete_absolute_error",
    ),
    complete_summary(
        "XGBoost Integrated Framework",
        "xgb_integrated_projected_latest_bixs",
        "xgb_complete_absolute_error",
    ),
    complete_summary(
        "Incurred Chain Ladder",
        "cl_projected_latest_bixs_T1",
        "cl_complete_absolute_error",
    ),
])

complete_portfolio_summary.to_csv(
    OUTPUT_FOLDER / "historical_complete_framework_portfolio_summary.csv",
    index=False,
)

display(complete_portfolio_summary)

## Compare the one-year and two-year tests on a common scope

When the one-year Integrated Framework output is available, this section restricts both
historical tests to their shared accident years and recomputes the same portfolio measures.
This separates the effect of a longer projection horizon from changes in accident-year scope.

In [ ]:
one_year_ay_path = (
    ONE_YEAR_INTEGRATED_FOLDER
    / "historical_integrated_framework_by_accident_year.csv"
)

if one_year_ay_path.exists():
    one_year_ay = pd.read_csv(one_year_ay_path)

    common_ays = sorted(
        set(one_year_ay["ACC_YEAR"].astype(int))
        .intersection(set(integrated_ay["ACC_YEAR"].astype(int)))
    )

    method_columns = {
        "Random Forest Integrated Framework": (
            "rf_integrated_projected_latest_bixs",
            "full_portfolio_actual_latest_bixs_T1",
        ),
        "XGBoost Integrated Framework": (
            "xgb_integrated_projected_latest_bixs",
            "full_portfolio_actual_latest_bixs_T1",
        ),
        "Incurred Chain Ladder": (
            "cl_projected_latest_bixs_T1",
            "full_portfolio_actual_latest_bixs_T1",
        ),
    }

    horizon_rows = []

    for horizon_name, horizon_df, horizon_t0, horizon_t1 in [
        (
            "One-year",
            one_year_ay.loc[one_year_ay["ACC_YEAR"].isin(common_ays)].copy(),
            str(one_year_ay["T0"].iloc[0]) if "T0" in one_year_ay else "2021 Q4",
            str(one_year_ay["T1"].iloc[0]) if "T1" in one_year_ay else "2022 Q4",
        ),
        (
            "Two-year",
            integrated_ay.loc[integrated_ay["ACC_YEAR"].isin(common_ays)].copy(),
            T0_LABEL,
            T1_LABEL,
        ),
    ]:
        for method, (projected_col, observed_col) in method_columns.items():
            observed = float(horizon_df[observed_col].sum())
            projected = float(horizon_df[projected_col].sum())
            errors = horizon_df[projected_col] - horizon_df[observed_col]

            horizon_rows.append({
                "horizon": horizon_name,
                "T0": horizon_t0,
                "T1": horizon_t1,
                "method": method,
                "common_accident_year_min": min(common_ays),
                "common_accident_year_max": max(common_ays),
                "observed_latest_bixs_T1": observed,
                "projected_latest_bixs_T1": projected,
                "difference_projected_minus_observed": projected - observed,
                "bias_ratio": projected / observed if observed != 0 else np.nan,
                "accident_year_wape": (
                    float(errors.abs().sum()) / abs(observed)
                    if observed != 0
                    else np.nan
                ),
            })

    horizon_comparison = pd.DataFrame(horizon_rows)
    horizon_comparison.to_csv(
        OUTPUT_FOLDER / "historical_horizon_comparison_common_scope.csv",
        index=False,
    )
    display(horizon_comparison)
else:
    warnings.warn(
        "One-year Integrated Framework output was not found. "
        "The two-year results are complete, but the optional common-scope "
        "horizon comparison was skipped."
    )

## Run the final audit

This section records the key control totals for the two-year Integrated Framework,
including estimated versus observed pure IBNR, reconciliation differences and the AY2020
early-maturity result. The saved audit provides a final check before the results are used in
the dissertation.

In [ ]:
audit_summary = pd.DataFrame([{
    "T0": T0_LABEL,
    "T1": T1_LABEL,
    "rolling_comparator_years": ROLLING_COMPARATOR_YEARS,
    "main_accident_year_min": min(TARGET_AYS),
    "main_accident_year_max": max(TARGET_AYS),
    "current_accident_year": CURRENT_AY,
    "estimated_pure_ibnr_main_total": float(
        pure_main["estimated_pure_ibnr_bixs_amount"].sum()
    ),
    "observed_pure_ibnr_main_total": float(
        pure_main["observed_pure_ibnr_bixs_amount_T1"].sum()
    ),
    "pure_ibnr_total_difference": float(
        pure_main["estimated_pure_ibnr_bixs_amount"].sum()
        - pure_main["observed_pure_ibnr_bixs_amount_T1"].sum()
    ),
    "maximum_amount_reconciliation_difference": max_amount_difference,
    "current_ay_macro_projection": early_projected_latest,
    "current_ay_observed_latest": early_observed_latest,
    "current_ay_macro_error": early_projected_latest - early_observed_latest,
}])

audit_summary.to_csv(
    OUTPUT_FOLDER / "historical_integrated_framework_audit_summary.csv",
    index=False,
)

display(audit_summary.T)

print("Saved all outputs to:", OUTPUT_FOLDER)

## Main output files

The notebook saves the comparator detail, pure IBNR estimates and reconciliation audit,
the accident-year and portfolio Integrated Framework results, the AY2020 early-maturity
results, the complete portfolio comparison, and the optional common-scope horizon
comparison in `OUTPUT_FOLDER`.